# ___Phylogenetic traversal of ACEs___
---------------------------------------------------

In [1]:
print(R.version$version.string)

[1] "R version 4.5.2 (2025-10-31 ucrt)"


In [2]:
suppressPackageStartupMessages({
    library("ape")
    library("phytools")
    library("corHMM")
})

In [3]:
STATES <- read.csv("../../data/chapter2/FREDv3subset/finalized_states_395_species.csv", stringsAsFactors = TRUE)[, c("binominal", "state")] # finalized mycorrhizal states
COLLAB_AXIS <- read.csv("../../data/chapter2/FREDv3subset/collab_ord1_species_avgs_SRL_RD.csv", stringsAsFactors = TRUE) # first order species averaged RD and SRL values
MERGED <- merge(x = STATES, y = COLLAB_AXIS, by = "binominal") # merge the two datasets into one, based on the binominal names
stopifnot(nrow(MERGED)==395)

PHYLOGENY <- ape::multi2di(ape::read.tree("../../data/chapter2/uphylomaker/FRED_subset_collab_395sp.tre")) # phylogenetic tree created for the 395 species using U.PhyloMaker
stopifnot(length(PHYLOGENY$tip.label)==395)

# MERGED contains spaces in the binominal names - replace that with underscores; and '/' in mycorrhizal states that need to be removed
data <- data.frame(binominal = gsub(MERGED$binominal, pattern = ' ', replacement = '_'), RD = MERGED$F00679, SRL = MERGED$F00727, myco = gsub(x = MERGED$state, pattern = '/', replacement = ''))
matched_row_indices <- match(PHYLOGENY$tip.label, data$binominal)
stopifnot(all(data$binominal[matched_row_indices] == PHYLOGENY$tip.label))
data <- data[matched_row_indices, ] # reorder the dataset to match the species order in the phylogeny
stopifnot(all(data$binominal == PHYLOGENY$tip.label))
stopifnot(length(unique(data$binominal)) == length(data$binominal))

In [4]:
# FOR CONVENIRNCE
RD <- setNames(object = data$RD, nm = data$binominal)
SRL <- setNames(object = data$SRL, nm = data$binominal)
STATES <- setNames(object = data$myco, nm = data$binominal)

In [8]:
# PLOT THE PHYLOGENY AND NODE & TIP NUMBERS, IT'LL HELP IN TRACING DOWN CHANGES AND SHIFTS IN TRAITS
par(mar=c(0, 0, 0, 0))
png(filename = "../../plots/FRED_subset_collab_395sp_nodes_n_tips.png", width = 18000, height = 18000, units = "px", res = 300)
phytools::plotTree(tree = PHYLOGENY, ftype = "i", fsize = 1.2, type = "fan", lwd = 1, offset = 4)
# with interactive left to default, got a warning for every label
phytools::labelnodes(text = 1:(PHYLOGENY$Nnode + length(PHYLOGENY$tip.label)), node = 1:(PHYLOGENY$Nnode + length(PHYLOGENY$tip.label)), cex = 1, interactive = FALSE)
dev.off()

agg_record_1770286312 
                    2

In [5]:
#-----------------------------------------------
# ACE OF DISCRETE CATEGORICAL TRAITS
#-----------------------------------------------

# choosing the ARD model for mycorrhizal state evolution as it has been demonstrated that different transitions happen at different rates
# and some transitions are prectically irreversible compared to others

# ace_mystates_rr <- phytools::rerootingMethod(tree = PHYLOGENY, x = STATES, model = "ARD")
ace_mystates_hmm <- corHMM::corHMM(phy = PHYLOGENY, data = data[, c("binominal", "myco")], model = "ARD", node.states = "marginal", rate.cat = 1)

#--------------------------------
# ACE OF CONTINUOUS TRAITS
#--------------------------------

ace_srl_fastanc <- phytools::fastAnc(tree = PHYLOGENY, x = SRL)

You specified 'fixed.nodes=FALSE' but included a phy object with node labels. These node labels have been removed.


Warning message in corHMM::corHMM(phy = PHYLOGENY, data = data[, c("binominal", :
"Branch lengths of 0 detected. Adding 1e-5 to these branches."


State distribution in data:
States:	1	2	3	4	5	6	
Counts:	300	15	8	65	3	4	
Beginning thorough optimization search -- performing 0 random restarts 
Finished. Inferring ancestral states using marginal reconstruction. 


In [ ]:
# WE DO HAVE POLYTOMIES IN OUR PHYLOGENY :(

In [6]:
ace_mystates_hmm # WE DO SEE TRANSITION RATE DIFFERENCES 


Fit
      -lnL      AIC     AICc Rate.cat ntax
 -127.3412 314.6824 319.7923        1  395

Legend
      1       2       3       4       5       6 
   "AM" "AMEcM"  "AMNM"   "EcM"   "ErM"    "NM" 

Rates
             (1,R1)      (2,R1)       (3,R1)       (4,R1)       (5,R1)
(1,R1)           NA 0.000322027 0.0009748366 0.0002015374 6.729752e-05
(2,R1) 1.000336e-09          NA 0.0000000010 0.0174455509 1.000000e-09
(3,R1) 1.691330e-02 0.000000001           NA 0.0000000010 1.000000e-09
(4,R1) 1.299848e-03 0.000000001 0.0000000010           NA 1.000000e-09
(5,R1) 1.000000e-09 0.000000001 0.0000000010 0.0000000010           NA
(6,R1) 1.000000e-09 0.000000001 0.0071676917 0.0000000010 1.000000e-09
             (6,R1)
(1,R1) 0.0000000010
(2,R1) 0.0000000010
(3,R1) 0.0102782964
(4,R1) 0.0000000010
(5,R1) 0.0009739048
(6,R1)           NA

Arrived at a reliable solution 

In [23]:
head(ace_mystates_hmm$states) # probabilities of each internal node of the phylogeny belonging to the given states

"(1,R1)","(2,R1)","(3,R1)","(4,R1)","(5,R1)","(6,R1)"
0.7384076,1.040770e-04,0.0993487387,6.261923e-03,6.054074e-04,1.552723e-01
0.8500365,7.905156e-04,0.0746536180,5.613651e-03,2.862179e-05,6.887709e-02
0.9991832,9.825375e-13,0.0008165587,7.341792e-08,1.816180e-15,1.231038e-07
0.9992081,2.686675e-11,0.0007917985,3.029895e-08,2.461497e-14,8.734257e-08
0.9989132,2.536349e-10,0.0010865789,5.341592e-08,3.150591e-12,2.026010e-07
0.9990792,3.114671e-09,0.0009205367,1.242975e-07,2.426444e-12,1.727226e-07


In [29]:
rowSums(ace_mystates_hmm$states) # probabilities of all states add up to 1.00 :)

[1] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 [38] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 [75] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[112] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[149] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[186] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[223] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[260] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[297] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[334] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[371] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1

In [31]:
# figure out which state is the most probable for all the internal nodes
apply(ace_mystates_hmm$states, MARGIN = 1, FUN = which.max) # MARGIN = 1 means apply the function to each row

[1] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 [38] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 5 5 1 1 1 1 1 1 1 1
 [75] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 1 1 1 1 1 1 1
[112] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 4
[149] 4 4 4 4 4 4 4 4 4 4 4 4 4 4 1 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
[186] 4 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 4
[223] 4 4 4 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[260] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 3 3 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[297] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[334] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 4 4 4 4 4 4 4 4 4 4
[371] 4 4 4 4 4 4 4 4 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 6

In [38]:
PHYLOGENY # our phylogeny has 395 tips ans 394 internal nodes and we reconstructed ancestral traits for the internal nodes


Phylogenetic tree with 395 tips and 394 internal nodes.

Tip labels:
  Anaphalis_aureopunctata, Anaphalis_hancockii, Solidago_decurrens, Doellingeria_scabra, Aster_tataricus, Artemisia_igniaria, ...
Node labels:
  , Spermatophyta, Mesangiospermae, mrcaott2ott121, eudicotyledons, mrcaott2ott969, ...

Rooted; includes branch length(s).

In [39]:
dim(ace_mystates_hmm$states) # got 394 rows and 6 columns

[1] 394   6

In [ ]:
# reconstructed ancestral states of the internal nodes, with node numbers as names
setNames(colnames(ace_mystates_rr$marginal.anc)[apply(ace_mystates_rr$marginal.anc, MARGIN = 1, FUN=which.max)],
         nm = rownames(ace_mystates_rr$marginal.anc))

In [ ]:


# WHAT WE NEED HERE IS A DICTIONARY OF WHICH NODES DESCEND FROM WHICH NODES, SO WE CAN LOOK UP THE STATE TRANSITIONS AND CONTINUOUS TRAIT CHANGES
# http://www.phytools.org/eqg/Exercise_3.2/
# By convention, the tips of the tree are numbered 1 through n for n tips; and the nodes are numbered n + 1 through n + m for m nodes
# the matrix edge contains the beginning and ending node number for all the nodes and tips in the tree.
PHYLOGENY$edge

phyedges <- as.data.frame(PHYLOGENY$edge)
colnames(phyedges) <- c("from", "to")

for (i in 1:nrow(phyedges)) {
    print(ace_srl_fastanc[as.character(phyedges[i, ][, "from"])]) # - ace_srl_fastanc[phyedges[i, ][, "to"]])
}

